KPI 3 - Error Rate


Loading libararies 

In [1]:
pip install statsmodels


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Desmond\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest

Loading data sets 

In [3]:
import pandas as pd

data_path = r"c:\Users\Desmond\Desktop\AB testing Project\Data"

exp    = pd.read_csv(f"{data_path}\\df_final_experiment_clients_cleaned.csv")
funnel = pd.read_csv(f"{data_path}\\web_funnel_sorted.csv")

print("exp shape:", exp.shape)
print("funnel shape:", funnel.shape)

exp shape: (70609, 2)
funnel shape: (744641, 6)


Confrimation Check 

In [4]:
print(funnel.shape)
print(funnel.columns.tolist())
print(funnel["process_step"].unique())  # confirm step names
print(exp["group_test"].unique())       # confirm "Test" / "Control" labels

(744641, 6)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'source']
<StringArray>
['step_3', 'start', 'step_1', 'step_2', 'confirm']
Length: 5, dtype: str
<StringArray>
['test', 'control', nan]
Length: 3, dtype: str


Hypothesis:

The New  UI has the same or more errors than the  Old UI 


The New  UI has fewer errors than than Old UI

- H₀: error_rate(Test) ≥ error_rate(Control)   
- H₁: error_rate(Test) < error_rate(Control)   

Error  is defiend as a backward step in the funnel, movingin backwards from step 2 to step 1 indicating user cofusion or friction 

Detecting Errors 

 Sorting  by client and timestamp to preserve step order

 Mapping  steps to numeric order


Detecting Errors 

In [5]:
funnel_sorted = funnel.sort_values(["client_id", "date_time"]).copy()

step_order = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
funnel_sorted["step_num"] = funnel_sorted["process_step"].map(step_order)

funnel_sorted["prev_step"] = funnel_sorted.groupby("client_id")["step_num"].shift(1)
funnel_sorted["is_error"]  = funnel_sorted["step_num"] < funnel_sorted["prev_step"]

In [6]:
client_errors = (
    funnel_sorted.groupby("client_id")["is_error"]
    .any()
    .reset_index()
    .rename(columns={"is_error": "had_error"})
)

client_errors = client_errors.merge(
    exp[["client_id", "group_test"]], on="client_id", how="inner"
)

In [7]:
print(exp["group_test"].unique())
print(exp["group_test"].value_counts())

<StringArray>
['test', 'control', nan]
Length: 3, dtype: str
group_test
test       26968
control    23532
Name: count, dtype: int64


In [8]:
print(client_errors.shape)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

(70609, 3)
group_test
test       26968
control    23532
Name: count, dtype: int64
had_error
False    45114
True     25495
Name: count, dtype: int64


 One row per client: did they make at least one error?

In [9]:
client_errors = (
    funnel_sorted.groupby("client_id")["is_error"]
    .any()
    .reset_index()
    .rename(columns={"is_error": "had_error"})
)





Merge with experiment group labels

In [10]:
client_errors = client_errors.merge(
    exp[["client_id", "group_test"]], on="client_id", how="inner"
)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

group_test
test       26968
control    23532
Name: count, dtype: int64
had_error
False    45114
True     25495
Name: count, dtype: int64


Group by Error rates 

In [11]:
error_rates = (
    client_errors.groupby("group_test")["had_error"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "errors", "count": "total", "mean": "error_rate"})
)

print(error_rates)



            errors  total  error_rate
group_test                           
control       8089  23532    0.343745
test         10195  26968    0.378041


Testsing 

In [12]:
from statsmodels.stats.proportion import proportions_ztest

test_grp    = client_errors[client_errors["group_test"] == "test"]
control_grp = client_errors[client_errors["group_test"] == "control"]

count = [test_grp["had_error"].sum(), control_grp["had_error"].sum()]
nobs  = [len(test_grp), len(control_grp)]


# alternative='smaller' → H₁: Test error rate < Control error rate
z_stat, p_value = proportions_ztest(count, nobs, alternative="smaller")

print(f"Z-statistic : {z_stat:.4f}")
print(f"P-value     : {p_value:.4f}")
print(f"Alpha       : 0.05")
print()
if p_value < 0.05:
    print("Reject H₀ — Test group has a significantly LOWER error rate.")
else:
    print("Fail to reject H₀ — No significant reduction in error rate.")


Z-statistic : 7.9996
P-value     : 1.0000
Alpha       : 0.05

Fail to reject H₀ — No significant reduction in error rate.


Final Conclusion 

The result supports H₀ — the Test group had a higher error rate than Control, meaning the new UI did not reduce errors and may have actually increased user friction.

Creating  Summary table

In [19]:
error_summary = (
    client_errors.groupby("group_test")["had_error"]
    .agg(
        total_clients   = "count",
        error_count     = "sum",
        no_error_count  = lambda x: (~x).sum(),
        error_rate      = "mean"
    )
    .reset_index()
)

error_summary["error_rate_pct"] = (error_summary["error_rate"] * 100).round(2)

print(error_summary)
error_summary.to_csv(
    r"c:\Users\Desmond\Desktop\AB testing Project\Data\kpi3_error_rate_summary.csv",
    index=False
)
print("Saved successfully")

  group_test  total_clients  error_count  no_error_count  error_rate  \
0    control          23532         8089           15443    0.343745   
1       test          26968        10195           16773    0.378041   

   error_rate_pct  
0           34.37  
1           37.80  
Saved successfully


Exporting 

In [22]:
import os

save_path = os.path.expanduser(r"~\Desktop")

error_summary.to_csv(f"{save_path}\\kpi3_error_rate_summary.csv", index=False)
client_errors.to_csv(f"{save_path}\\kpi3_client_errors.csv", index=False)
print("All files saved to Desktop")

All files saved to Desktop


Client Level Detail 

In [23]:
client_errors.to_csv(f"{save_path}\\kpi3_client_errors.csv", index=False)

Statistical results:

In [ ]:
stats_result = pd.DataFrame({
    "metric" : ["Z-statistic", "P-value", "Alpha", "Result"],
    "value"  : [7.9996, 1.0000, 0.05, "Fail to Reject H0"]
})
stats_result.to_csv(f"{save_path}\\kpi3_stats_result.csv", index=False)

print("Saved to Desktop:")
print(f"  {save_path}\\kpi3_error_rate_summary.csv")
print(f"  {save_path}\\kpi3_client_errors.csv")
print(f"  {save_path}\\kpi3_stats_result.csv")

In [24]:
import os

save_path = os.path.expanduser(r"~\Desktop")

files = [
    "kpi3_error_rate_summary.csv",
    "kpi3_client_errors.csv",
    "kpi3_stats_result.csv"
]

for f in files:
    full_path = f"{save_path}\\{f}"
    exists = os.path.exists(full_path)
    print(f"{f}: {'✅ Found' if exists else '❌ Missing'}")

kpi3_error_rate_summary.csv: ✅ Found
kpi3_client_errors.csv: ✅ Found
kpi3_stats_result.csv: ❌ Missing


In [25]:
import pandas as pd
import os

save_path = os.path.expanduser(r"~\Desktop")

stats_result = pd.DataFrame({
    "metric" : ["Z-statistic", "P-value", "Alpha", "Result"],
    "value"  : [7.9996, 1.0000, 0.05, "Fail to Reject H0"]
})

stats_result.to_csv(f"{save_path}\\kpi3_stats_result.csv", index=False)
print("kpi3_stats_result.csv saved to Desktop ✅")

kpi3_stats_result.csv saved to Desktop ✅
